# [TODO: Your Project Title Here]
## CMSC 320 — Introduction to Data Science | Summer 2026

---

### Contributions

| Member | Sections | Summary |
|--------|----------|---------|
| Nipun Maisheri | A, B | Wrote the Introduction section (research questions, motivation, approach), and the Data Curation section detailing the dataset source, size, and preprocessing scope. |
| [TODO: Name 2] | C, D | [TODO: 1–2 sentences describing specific contributions] |
| [TODO: Name 3] | E, F | [TODO: 1–2 sentences describing specific contributions] |
| [TODO: Name 4] | E, F, H | [TODO: 1–2 sentences describing specific contributions] |

**Section Key:**
- A: Project idea  
- B: Dataset Curation and Preprocessing  
- C: Data Exploration and Summary Statistics  
- D: ML Algorithm Design/Development  
- E: ML Algorithm Training and Test Data Analysis  
- F: Visualization, Result Analysis, Conclusion  
- G: Final Tutorial Report Creation  
- H: Additional (not listed above)

---
## 1. Introduction

This project analyzes the [NYC Motor Vehicle Collisions — Crashes dataset](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95), a public record of over 2.2 million police-reported crashes across New York City maintained by the NYPD via NYC Open Data. Each row documents a single crash, including its date and time, borough, involved vehicle types, contributing factors, and the number of people injured or killed. Motor vehicle collisions are one of the leading causes of preventable injury and death in urban areas, and NYC's scale and data-collection history make it a rich case study for understanding when, where, and why crashes turn severe.

Our central research question is: **can we predict whether a crash will be fatal, and what factors most strongly drive that risk?** We break this down into three supporting questions explored through hypothesis testing in Checkpoint 2 and carried forward here: (1) Is a crash's fatality outcome associated with the type of vehicle involved? (2) Do crashes during rush hour differ in average injury severity from crashes at other times of day? (3) Are crashes at the city's highest-accident intersections more likely to be fatal than crashes elsewhere? We then build on these findings with a machine learning model that predicts fatal crashes directly from crash-time features.

These questions matter because the answers have direct, actionable value: city planners and the NYPD can use insight into high-risk vehicle types, times, and locations to target enforcement, infrastructure changes, or public-safety campaigns; insurers can better understand risk factors; and drivers and cyclists can be more informed about when and where collisions are most dangerous. Understanding fatality risk is more urgent than understanding crash frequency alone, since resources for preventing deaths are limited and should be directed where they have the greatest impact.

Our approach proceeds in three stages. First, we clean and preprocess the raw NYC Open Data export (2.2M+ rows, 37 columns), standardizing types and deriving fields such as crash hour and a binary fatal-crash indicator. Second, we perform exploratory data analysis using statistical hypothesis tests (chi-square, t-test, and Mann-Whitney U) to test the three questions above. Third, we frame fatal-crash prediction as a binary classification problem and train Logistic Regression and Random Forest models, evaluating them with confusion matrices, feature importances, and ROC curves to identify which factors most influence crash fatality.

---
## 2. Data Curation

The data comes from the [NYC Motor Vehicle Collisions — Crashes dataset](https://data.cityofnewyork.us/Public-Safety/Motor-Vehicle-Collisions-Crashes/h9gi-nx95) published by the New York City Police Department (NYPD) via NYC OpenData. The dataset records every police-reported motor vehicle collision in NYC. Each row represents a single crash event and includes information such as crash date and time, borough, street location, number of persons injured/killed, contributing factors, and vehicle types involved.

The raw export is retrieved directly from the NYC Open Data Socrata API as a JSON file (`rows.json`, roughly 1 GB), rather than committed to this repository, since it far exceeds GitHub's 100 MB file-size limit. The dataset covers every reported collision from **July 2012 to the present**, with new crashes added continuously — our pipeline downloads the current snapshot at run time, so the exact row count grows slightly with each run. At the time we developed this analysis, the snapshot contained **approximately 2.27 million rows across 37 columns** (after dropping 8 Socrata bookkeeping columns, 33 usable columns remain). A small minority of crashes in the dataset resulted in at least one fatality (the exact rate is computed in the preprocessing cell below), reflecting the substantial class imbalance we account for later in the modeling stage.

In [ ]:
# ── Imports ────────────────────────────────────────────────────────────────────
import os
import json
import urllib.request
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import ttest_ind, mannwhitneyu, chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, RocCurveDisplay
)

# ── Load ───────────────────────────────────────────────────────────────────────
# The raw file is ~1 GB — too large to commit to GitHub (100 MB limit). Instead,
# we download it directly from the NYC Open Data API the first time this runs
# and cache it locally as rows.json (gitignored). Later runs reuse the cache.
DATA_URL = "https://data.cityofnewyork.us/api/views/h9gi-nx95/rows.json?accessType=DOWNLOAD"
PATH = "rows.json"

if not os.path.exists(PATH):
    print("rows.json not found locally — downloading from NYC Open Data (~1 GB, this may take a few minutes)...")
    urllib.request.urlretrieve(DATA_URL, PATH)
    print("Download complete.")
else:
    print("Using cached rows.json.")

with open(PATH) as f:
    raw = json.load(f)

columns = [c["fieldName"] for c in raw["meta"]["view"]["columns"]]
df = pd.DataFrame(raw["data"], columns=columns)
del raw

print(f"Raw shape: {df.shape}")
df.head(3)

In [ ]:
# ── Preprocessing (carried over from Checkpoint 2) ────────────────────────────
# Drop Socrata bookkeeping columns
df = df.drop(columns=[c for c in df.columns if c.startswith(":")])

# Standardize string values. select_dtypes(include="object") can include columns
# that are all-null (and therefore inferred as float, not string), so we only
# apply .str methods to columns pandas actually recognizes as string-typed.
text_cols = [
    c for c in df.select_dtypes(include="object").columns
    if pd.api.types.infer_dtype(df[c], skipna=True) == "string"
]
df[text_cols] = df[text_cols].apply(lambda col: col.str.strip().str.upper())

# Numeric conversions
for col in ["number_of_persons_injured", "number_of_persons_killed",
            "number_of_pedestrians_injured", "number_of_pedestrians_killed",
            "number_of_cyclist_injured", "number_of_cyclist_killed",
            "number_of_motorist_injured", "number_of_motorist_killed",
            "latitude", "longitude", "zip_code"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Datetime
df["crash_date"] = pd.to_datetime(df["crash_date"], errors="coerce")
df["hour"] = pd.to_datetime(df["crash_time"], format="%H:%M", errors="coerce").dt.hour

# Derived target variable
df["fatal_crash"] = (df["number_of_persons_killed"] > 0).astype(int)

print(f"Clean shape: {df.shape}")
print(f"Fatal crashes: {df['fatal_crash'].sum()} ({df['fatal_crash'].mean():.2%})")

---
## 3. Exploratory Data Analysis

**[TODO: Copy/paste (or re-run) the three hypothesis-testing sections from Checkpoint 2 below. Make sure to include the prose explanations for each conclusion.]**

We present three conclusions drawn from the data using statistical hypothesis testing.

### 3.1 Hypothesis 1 — Vehicle Type and Fatal Crash Risk

**[TODO: Paste CP2 Hypothesis 1 markdown + code cells here.]**

In [ ]:
# [TODO: Paste Hypothesis 1 code from Checkpoint 2]

**[TODO: Paste Hypothesis 1 conclusion paragraph from Checkpoint 2]**

### 3.2 Hypothesis 2 — Rush Hour vs. Non-Rush-Hour Injuries

**[TODO: Paste CP2 Hypothesis 2 markdown + code cells here.]**

In [ ]:
# [TODO: Paste Hypothesis 2 code from Checkpoint 2]

**[TODO: Paste Hypothesis 2 conclusion paragraph from Checkpoint 2]**

### 3.3 Hypothesis 3 — High-Accident Intersections and Fatality Rates

**[TODO: Paste CP2 Hypothesis 3 markdown + code cells here.]**

In [ ]:
# [TODO: Paste Hypothesis 3 code from Checkpoint 2]

**[TODO: Paste Hypothesis 3 conclusion paragraph from Checkpoint 2]**

---
## 4. Primary Analysis — Machine Learning

### 4.1 Problem Framing and Algorithm Choice

**[TODO: Write 1–2 paragraphs explaining your choice. Example framing below — adapt or replace entirely.]**

Our EDA showed that vehicle type, time of day, and intersection characteristics are all associated with whether a crash turns fatal. We therefore frame this as a **binary classification problem**: given features available at crash-report time, can we predict whether a crash results in at least one fatality (`fatal_crash = 1`)?

We select **Random Forest classification** because:
1. It handles the mixture of categorical (borough, vehicle type) and numeric (hour, injury count) features without extensive preprocessing.
2. Its built-in feature-importance scores will let us answer our research question about *which* factors most influence fatality.
3. It is robust to class imbalance compared to a plain logistic regression baseline.

We will also train a **Logistic Regression** baseline so we have a simple comparison point.

**[TODO: Add a link to a resource that explains Random Forests for readers who want to learn more — e.g., Towards Data Science or scikit-learn docs.]**

### 4.2 Feature Engineering

**[TODO: Add a prose explanation of each feature you choose and why.]**

In [ ]:
# ── Feature selection ─────────────────────────────────────────────────────────
# [TODO: Adjust the feature list to match what is clean and available in your df]
FEATURE_COLS = [
    "hour",
    "number_of_persons_injured",
    "number_of_pedestrians_injured",
    "number_of_cyclist_injured",
    "number_of_motorist_injured",
    "borough",
    "vehicle_type_code1",
    "contributing_factor_vehicle_1",
]
TARGET_COL = "fatal_crash"

ml_df = df[FEATURE_COLS + [TARGET_COL]].dropna()
print(f"ML dataset shape: {ml_df.shape}")
print(f"Class balance:\n{ml_df[TARGET_COL].value_counts(normalize=True)}")

In [ ]:
# ── Encode categoricals ───────────────────────────────────────────────────────
cat_cols = ["borough", "vehicle_type_code1", "contributing_factor_vehicle_1"]

# Keep only the top-K categories for each column to limit dimensionality
# [TODO: Adjust K as needed]
K = 20
for col in cat_cols:
    top_k = ml_df[col].value_counts().head(K).index
    ml_df[col] = ml_df[col].where(ml_df[col].isin(top_k), other="OTHER")

ml_encoded = pd.get_dummies(ml_df, columns=cat_cols, drop_first=True)

X = ml_encoded.drop(columns=[TARGET_COL])
y = ml_encoded[TARGET_COL]

print(f"Feature matrix shape: {X.shape}")

### 4.3 Train / Test Split

We hold out 20 % of the data as a test set. The split is stratified to preserve the class ratio.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]} rows | Test: {X_test.shape[0]} rows")

### 4.4 Model Training

In [ ]:
# ── Logistic Regression baseline ─────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
lr.fit(X_train_scaled, y_train)

print("Logistic Regression — Test Set Results")
print(classification_report(y_test, lr.predict(X_test_scaled)))

In [ ]:
# ── Random Forest ─────────────────────────────────────────────────────────────
# [TODO: Tune n_estimators, max_depth, class_weight as needed]
rf = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)

print("Random Forest — Test Set Results")
print(classification_report(y_test, rf.predict(X_test)))

---
## 5. Visualization

**[TODO: Write a prose paragraph introducing each plot before it appears. Explain what the reader should take away.]**

In [ ]:
# ── Plot 1: Confusion matrix ──────────────────────────────────────────────────
# [TODO: Write 1–2 sentences explaining what the confusion matrix shows]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, model, name, X_ev in [
    (axes[0], lr, "Logistic Regression", X_test_scaled),
    (axes[1], rf, "Random Forest",       X_test),
]:
    ConfusionMatrixDisplay.from_estimator(
        model, X_ev, y_test,
        display_labels=["Non-Fatal", "Fatal"],
        ax=ax, colorbar=False
    )
    ax.set_title(name)

plt.tight_layout()
plt.suptitle("Confusion Matrices — Logistic Regression vs Random Forest", y=1.02)
plt.savefig("confusion_matrices.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 2: Feature importances (Random Forest) ───────────────────────────────
# [TODO: Write 1–2 sentences explaining what the feature importance plot shows]
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(10, 6))
top15.plot(kind="barh", ax=ax)
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("Top 15 Feature Importances — Random Forest")
ax.invert_yaxis()
plt.tight_layout()
plt.savefig("feature_importances.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
# ── Plot 3: ROC curve comparison ──────────────────────────────────────────────
# [TODO: Write 1–2 sentences explaining what the ROC curve shows]
fig, ax = plt.subplots(figsize=(7, 5))

RocCurveDisplay.from_estimator(lr, X_test_scaled, y_test, ax=ax, name="Logistic Regression")
RocCurveDisplay.from_estimator(rf, X_test,        y_test, ax=ax, name="Random Forest")

ax.set_title("ROC Curve — Fatal Crash Prediction")
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig("roc_curve.png", dpi=150, bbox_inches="tight")
plt.show()

---
## 6. Insights and Conclusions

**[TODO: Write 3–5 paragraphs addressing the points below. Delete these prompts when done.]**

- **Answer your research questions**: What did the ML model reveal about which factors predict crash fatality? Did the results confirm or surprise you relative to the EDA findings?
- **Model performance**: How well did the models perform (accuracy, precision, recall, AUC)? What are the limitations? (class imbalance, missing data, feature gaps)
- **Real-world implications**: What should city planners, law enforcement, or drivers take away? Are there specific boroughs, vehicle types, or times of day that warrant targeted intervention?
- **Future work**: What would you do with more time? (more features, deep learning, spatial analysis with `geopandas`, time-series forecasting, etc.)
- **Informed reader check**: After reading this tutorial, does a reader unfamiliar with NYC traffic data now understand the problem, the data, and the findings? Would a data-science-literate reader learn something new?